# Medical RAG from Scratch: Does a Fine-Tuned Embedding Model Give Better Answers?

**What we are building**
- A system that answers medical research questions with **yes / no / maybe**, using PubMed abstracts as evidence.
- The experiment: compare a **general embedding model** with one we **fine-tune on medical text**, and measure if retrieval and answers improve.

**The pipeline**

`Question → embed → search the abstracts (vector + keyword) → top chunks → LLM → yes / no / maybe`

**How to use this notebook**
- Runtime → Change runtime type → **T4 GPU**.
- Run cells **top to bottom**, one at a time. Read the note above each cell first.
- Total time: about 45–60 minutes for a full run.

---

### Concepts cheat sheet (read once)

| Concept | Simple meaning |
|---|---|
| **Embedding** | A sentence turned into a list of numbers. Similar meaning → numbers close together, like addresses on a meaning map. |
| **Cosine similarity** | How close two embeddings are. 1 = same meaning, 0 = unrelated. |
| **Chunking** | Cutting long text into small pieces so the model can read each one fully. |
| **Vector database** | A library that finds text by **meaning** (we use Chroma). |
| **BM25 (keyword search)** | Classic search that matches **exact words**. Great for drug and disease names. |
| **Hybrid search** | Combine vector + keyword rankings, like two judges whose scores are merged. |
| **RAG** | Find evidence first, then let the LLM answer from it. An open-book exam. |
| **Fine-tuning** | Giving a pretrained model extra lessons on your topic. |
| **Contrastive learning** | Pull a question close to its correct abstract, push it away from wrong ones. "This sock matches this sock." |
| **In-batch negatives** | In a batch of 32 pairs, each question's 31 other abstracts are automatically the "wrong answers". No extra labels needed. |
| **Hit@K** | Was the correct abstract in the top K results? |
| **MRR@10** | How high the correct abstract ranked: 1st = 1, 2nd = 0.5, 3rd = 0.33 … averaged over all questions. |

## Part 0 — Setup

### 0.1 Check the GPU
You should see a **Tesla T4**. If you see an error, switch the runtime to T4 GPU and run again.

In [1]:
!nvidia-smi

Wed Nov 18 03:04:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 0.2 Install libraries
- `sentence-transformers` → embedding models + fine-tuning
- `datasets` → download PubMedQA
- `chromadb` → vector database
- `rank_bm25` → keyword search
- `gradio` → simple web app

If Colab says **"restart session"** after this, click it, then continue from the next cell.

In [2]:
!pip install -q -U sentence-transformers datasets accelerate chromadb rank_bm25 gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
t

### 0.3 Log in to Hugging Face
**One-time prep:**
1. Create a free account at huggingface.co.
2. Open the page for **meta-llama/Llama-3.2-1B-Instruct** and click **request access** (approval is usually quick).
3. Settings → Access Tokens → create a **Read** token.
4. In Colab, click the 🔑 **Secrets** icon (left sidebar) → add `HF_TOKEN` with your token → turn on notebook access.

Why: Llama is a "gated" model; Hugging Face needs to know who you are.

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [4]:
from huggingface_hub import whoami
print(whoami()["name"])

Akshaya2112


### 0.4 Imports and settings
- `SEED` makes random steps repeatable, so you get the same results each run.
- `WANDB_DISABLED` stops training from asking for a Weights & Biases login.

In [5]:
import os, re, random
import numpy as np
import pandas as pd
import torch

os.environ["WANDB_DISABLED"] = "true"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

Using: cuda


### 0.5 (Optional) Save to Google Drive
Colab deletes files when the session ends. Run this if you want to keep the fine-tuned model and results.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/pubmed_rag"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/pubmed_rag


## Part 1 — Load and understand the data

**PubMedQA** has medical research questions, each paired with a PubMed abstract.
- `pqa_labeled` → 1,000 questions with expert answers (yes / no / maybe). **Our test set.**
- `pqa_unlabeled` → ~61,000 question–abstract pairs, no yes/no labels. **Our fine-tuning data.**

In [7]:
from datasets import load_dataset

labeled   = load_dataset("qiaojin/PubMedQA", "pqa_labeled",   split="train")
unlabeled = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")

print(labeled)
print(unlabeled)

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

pqa_unlabeled/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 66.0MB            

pqa_unlabeled/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61249 [00:00<?, ? examples/s]

Dataset({
    features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
    num_rows: 1000
})
Dataset({
    features: ['pubid', 'question', 'context', 'long_answer'],
    num_rows: 61249
})


### 1.1 Look at one example
Always look at real rows before modeling. Notice:
- `question` → what we ask
- `context["contexts"]` → the abstract, split into sections
- `final_decision` → the correct answer

In [8]:
ex = labeled[0]
print("QUESTION:", ex["question"])
print("\nABSTRACT (first 600 chars):", " ".join(ex["context"]["contexts"])[:600])
print("\nANSWER:", ex["final_decision"])

QUESTION: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

ABSTRACT (first 600 chars): Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dyna

ANSWER: yes


### 1.2 How balanced are the answers?
If most answers are "yes", a model that always says "yes" already looks decent. Note this number; it is your **"dumb baseline"**.

In [9]:
pd.Series(labeled["final_decision"]).value_counts(normalize=True).round(3)

,proportion
yes,0.552
no,0.338
maybe,0.110


### 1.3 Build a clean test table
One row per question: id, question, full abstract, correct label.

In [10]:
test_df = pd.DataFrame({
    "pubid":    [str(p) for p in labeled["pubid"]],
    "question": labeled["question"],
    "abstract": [" ".join(c["contexts"]) for c in labeled["context"]],
    "label":    labeled["final_decision"],
})
print(test_df.shape)
test_df.head()

(1000, 4)


,pubid,question,abstract,label
0,21645374,Do mitochondria play a role in remodelling lac...,Programmed cell death (PCD) is the regulated d...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,Assessment of visual acuity depends on the opt...,no
2,9488747,"Syncope during bathing in infants, a pediatric...",Apparent life-threatening events in infants ar...,yes
3,17208539,Are the long-term results of the transanal pul...,The transanal endorectal pull-through (TERPT) ...,no
4,10808977,Can tailored interventions increase mammograph...,Telephone counseling and tailored print commun...,yes


### 1.4 Prevent data leakage
**Leakage** = the model sees test data during training, which makes scores look better than they really are.
We remove any training pair whose abstract id also appears in the test set.

In [11]:
test_ids = set(test_df.pubid)
train_pool = unlabeled.filter(lambda r: str(r["pubid"]) not in test_ids)
print("Unlabeled:", len(unlabeled), "→ after removing test overlap:", len(train_pool))

Filter:   0%|          | 0/61249 [00:00<?, ? examples/s]

Unlabeled: 61249 → after removing test overlap: 61249


## Part 2 — Chunking

Our embedding model reads at most **128 tokens** (~90–100 words) at once; anything longer is cut off.
So we split each abstract into **100-word chunks with a 20-word overlap**.
- Overlap is like overlapping photos in a panorama, so no sentence gets lost at a cut.
- Each chunk remembers which abstract (`pubid`) it came from.

In [12]:
def chunk_text(text, chunk_words=100, overlap=20):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_words]))
        if start + chunk_words >= len(words):
            break
        start += chunk_words - overlap
    return chunks

rows = []
for _, r in test_df.iterrows():
    for i, ch in enumerate(chunk_text(r["abstract"])):
        rows.append({"chunk_id": f"{r.pubid}_{i}", "pubid": r.pubid, "text": ch})

chunks_df = pd.DataFrame(rows)
print(len(test_df), "abstracts →", len(chunks_df), "chunks")
chunks_df.head()

1000 abstracts → 2769 chunks


,chunk_id,pubid,text
0,21645374_0,21645374,Programmed cell death (PCD) is the regulated d...
1,21645374_1,21645374,PCD in plants. The following paper elucidates ...
2,21645374_2,21645374,dynamics were delineated into four categories ...
3,16418930_0,16418930,Assessment of visual acuity depends on the opt...
4,16418930_1,16418930,"with various eye disorders, among them 39 with..."


## Part 3 — Baseline embedding model

### 3.1 Load it and see embeddings in action
`all-MiniLM-L12-v2` is a small, general-purpose model trained on everyday web text.
The similarity matrix below should show the two diabetes sentences close together (high number) and the stock-market sentence far away.

In [13]:
from sentence_transformers import SentenceTransformer

BASE_MODEL = "sentence-transformers/all-MiniLM-L12-v2"
base_model = SentenceTransformer(BASE_MODEL, device=DEVICE)
print("Max tokens per text:", base_model.max_seq_length)

demo = ["What causes type 2 diabetes?",
        "Insulin resistance is a major driver of type 2 diabetes.",
        "The stock market fell sharply today."]
emb = base_model.encode(demo, normalize_embeddings=True)
print("Embedding shape:", emb.shape)          # 3 sentences × 384 numbers each
print(pd.DataFrame(emb @ emb.T, index=["diabetes Q", "diabetes A", "stocks"],
                   columns=["diabetes Q", "diabetes A", "stocks"]).round(2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Max tokens per text: 128
Embedding shape: (3, 384)
            diabetes Q  diabetes A  stocks
diabetes Q        1.00        0.72    0.07
diabetes A        0.72        1.00    0.05
stocks            0.07        0.05    1.00


### 3.2 Measure baseline retrieval
For every test question, search all chunks and check whether chunks from its **own** abstract come back on top.

The evaluator reports several metrics. Focus on:
- **accuracy@1 / @5 / @10** → "Was the right abstract found in the top 1 / 5 / 10?" (same as Hit@K)
- **mrr@10** → how high it ranked

This is the score the fine-tuned model must beat.

In [14]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

queries       = dict(zip(test_df.pubid, test_df.question))
corpus        = dict(zip(chunks_df.chunk_id, chunks_df.text))
relevant_docs = chunks_df.groupby("pubid")["chunk_id"].apply(set).to_dict()

ir_eval = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant_docs,
    accuracy_at_k=[1, 5, 10], precision_recall_at_k=[5, 10],
    mrr_at_k=[10], ndcg_at_k=[10], map_at_k=[10],
    name="pubmedqa", show_progress_bar=True,
)

def show(scores):
    if not isinstance(scores, dict):
        return scores
    return pd.Series({k.split("cosine_")[-1]: round(v, 4)
                      for k, v in scores.items() if "cosine" in k})

base_scores = ir_eval(base_model)
show(base_scores)

/tmp/ipykernel_5866/3698591413.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:04<00:00,  4.98s/it]


,0
accuracy@1,0.9670
accuracy@5,0.9880
accuracy@10,0.9930
precision@5,0.4364
precision@10,0.2334
recall@5,0.8000
recall@10,0.8517
ndcg@10,0.8501
mrr@10,0.9765
map@10,0.7818


## Part 4 — Fine-tune the embedding model (the heart of the project)

**Goal:** teach MiniLM medical language, so a medical question lands near its matching abstract.

**How:**
- Take 10,000 (question, abstract) pairs from the unlabeled set.
- Group them into batches of 32.
- **MultipleNegativesRankingLoss**: in each batch, the model learns "question 1 ↔ abstract 1 is the match; abstracts 2–32 are wrong" and repeats this for every question.
- Bigger batches = more wrong answers to learn from = usually better.

**Time:** about 3–5 minutes per epoch on a T4.

In [15]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

N_TRAIN = 10_000
sample = train_pool.shuffle(seed=SEED).select(range(N_TRAIN))

train_examples = [
    InputExample(texts=[r["question"], " ".join(r["context"]["contexts"])])
    for r in sample
]
print("Example question:", train_examples[0].texts[0])
print("Example abstract:", train_examples[0].texts[1][:300], "...")

/tmp/ipykernel_5866/1769444111.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, losses


Example question: Unipolar mania: a distinct entity or characteristic of manic preponderance?
Example abstract: It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipola ...


### 4.1 Train
- `EPOCHS = 1` is a quick first run. Try 2–3 later and compare; that comparison is a good experiment to report.
- `warmup_steps` starts with tiny learning steps, like warming up before a run.

In [16]:
ft_model = SentenceTransformer(BASE_MODEL, device=DEVICE)   # start from the same base model

train_loader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

EPOCHS = 1
ft_model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(train_loader) * EPOCHS),
    show_progress_bar=True,
)

FT_PATH = "mini-pubmedqa-finetuned"
ft_model.save(FT_PATH)
print("Saved to", FT_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to mini-pubmedqa-finetuned


In [17]:
ft_model = SentenceTransformer(BASE_MODEL, device=DEVICE)   # start from the same base model

train_loader = DataLoader(train_examples, shuffle=True, batch_size=32)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

EPOCHS = 2
ft_model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=int(0.1 * len(train_loader) * EPOCHS),
    show_progress_bar=True,
)

FT_PATH = "mini-pubmedqa-finetuned"
ft_model.save(FT_PATH)
print("Saved to", FT_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.027171


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to mini-pubmedqa-finetuned


### 4.2 (Optional) Save to Drive and Hugging Face Hub
Pushing to the Hub gives you a public model link to put on your resume.
Your HF token needs **Write** permission for the push.

In [18]:
ft_model.save(f"{SAVE_DIR}/{FT_PATH}")          # needs Part 0.5

# Uncomment and add your username to publish:
# ft_model.push_to_hub("YOUR_HF_USERNAME/mini-pubmedqa-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Part 5 — Did fine-tuning help retrieval?

Same test, same corpus, same questions. The **only** thing that changed is the embedding model.
This "change one thing at a time" design is what makes it a fair experiment.

In [19]:
ft_scores = ir_eval(ft_model)

compare = pd.DataFrame({"baseline": show(base_scores), "fine_tuned": show(ft_scores)})
compare["change"] = (compare.fine_tuned - compare.baseline).round(4)
compare

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:05<00:00,  5.91s/it]


,baseline,fine_tuned,change
accuracy@1,0.9670,0.9770,0.0100
accuracy@5,0.9880,0.9910,0.0030
accuracy@10,0.9930,0.9940,0.0010
precision@5,0.4364,0.4434,0.0070
precision@10,0.2334,0.2360,0.0026
recall@5,0.8000,0.8114,0.0114
recall@10,0.8517,0.8613,0.0096
ndcg@10,0.8501,0.8613,0.0112
mrr@10,0.9765,0.9836,0.0071
map@10,0.7818,0.7960,0.0142


**Write down:** accuracy@1, accuracy@5, mrr@10 for both models.
Small improvements are normal: the base model is already decent, and PubMedQA questions often reuse words from their abstracts.

## Part 6 — Store chunks in a vector database (Chroma)

We create **one collection per embedding model**, with the same chunks but different embeddings.
`hnsw:space = cosine` tells Chroma to use cosine similarity.

In [20]:
import chromadb

chroma = chromadb.Client()
models = {"baseline": base_model, "fine_tuned": ft_model}
collections = {}

for name, m in models.items():
    col_name = f"pubmed_{name}"
    try:
        chroma.delete_collection(col_name)       # clean start if you re-run
    except Exception:
        pass
    col = chroma.create_collection(name=col_name, metadata={"hnsw:space": "cosine"})

    embs = m.encode(chunks_df.text.tolist(), normalize_embeddings=True,
                    batch_size=64, show_progress_bar=True)

    B = 1000                                      # add in batches
    for i in range(0, len(chunks_df), B):
        col.add(
            ids=chunks_df.chunk_id[i:i+B].tolist(),
            documents=chunks_df.text[i:i+B].tolist(),
            embeddings=embs[i:i+B].tolist(),
            metadatas=[{"pubid": p} for p in chunks_df.pubid[i:i+B]],
        )
    collections[name] = col
    print(name, "→", col.count(), "chunks stored")

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

baseline → 2769 chunks stored


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

fine_tuned → 2769 chunks stored


### 6.1 Vector search
Embed the question with the **same** model used for the chunks, then ask Chroma for the closest chunks.

In [21]:
def vector_search(question, model_name, k=10):
    q_emb = models[model_name].encode([question], normalize_embeddings=True).tolist()
    res = collections[model_name].query(query_embeddings=q_emb, n_results=k)
    return [{"chunk_id": cid, "pubid": meta["pubid"], "text": doc}
            for cid, meta, doc in zip(res["ids"][0], res["metadatas"][0], res["documents"][0])]

q = test_df.question[0]
print("Q:", q, "| correct pubid:", test_df.pubid[0])
for r in vector_search(q, "fine_tuned", k=3):
    print(r["pubid"], "→", r["text"][:120], "...")

Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? | correct pubid: 21645374
21645374 → Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascarien ...
21645374 → dynamics were delineated into four categories (M1-M4) based on characteristics including distribution, motility, and mem ...
21645374 → PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in ...


## Part 7 — Keyword search and hybrid search

**Why hybrid?** Vector search understands meaning ("heart attack" ≈ "myocardial infarction"), while keyword search nails exact terms (drug names, gene names). Medicine needs both.

**Reciprocal Rank Fusion (RRF):** each search gives a ranking. A chunk earns `1 / (60 + rank)` from each list, and we add the points. A chunk ranked high by **both** judges wins.

In [22]:
from rank_bm25 import BM25Okapi

def tokenize(t):
    return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenize(t) for t in chunks_df.text])

def keyword_search(question, k=10):
    scores = bm25.get_scores(tokenize(question))
    top = np.argsort(scores)[::-1][:k]
    return [{"chunk_id": chunks_df.chunk_id[i], "pubid": chunks_df.pubid[i], "text": chunks_df.text[i]}
            for i in top]

def hybrid_search(question, model_name, k=10, rrf_k=60):
    pool = max(30, k)
    fused = {}
    for results in (vector_search(question, model_name, pool), keyword_search(question, pool)):
        for rank, r in enumerate(results):
            entry = fused.setdefault(r["chunk_id"], {"doc": r, "score": 0.0})
            entry["score"] += 1.0 / (rrf_k + rank + 1)
    ranked = sorted(fused.values(), key=lambda x: x["score"], reverse=True)
    return [x["doc"] for x in ranked[:k]]

### 7.1 Compare all five retrieval setups
We score at the **abstract** level: several chunks from the same abstract count once.

In [23]:
def eval_retriever(search_fn, k=10):
    hit1 = hit5 = hit10 = 0
    rr = 0.0
    for q, pid in zip(test_df.question, test_df.pubid):
        ranked = []
        for r in search_fn(q, k * 3):              # fetch extra, then dedupe abstracts
            if r["pubid"] not in ranked:
                ranked.append(r["pubid"])
        ranked = ranked[:k]
        if pid in ranked:
            pos = ranked.index(pid) + 1
            rr += 1 / pos
            hit1 += pos <= 1; hit5 += pos <= 5; hit10 += pos <= 10
    n = len(test_df)
    return {"hit@1": hit1/n, "hit@5": hit5/n, "hit@10": hit10/n, "MRR@10": rr/n}

retrieval_results = {
    "keyword (BM25)":    eval_retriever(lambda q, k: keyword_search(q, k)),
    "vector baseline":   eval_retriever(lambda q, k: vector_search(q, "baseline", k)),
    "vector fine-tuned": eval_retriever(lambda q, k: vector_search(q, "fine_tuned", k)),
    "hybrid baseline":   eval_retriever(lambda q, k: hybrid_search(q, "baseline", k)),
    "hybrid fine-tuned": eval_retriever(lambda q, k: hybrid_search(q, "fine_tuned", k)),
}
retrieval_table = pd.DataFrame(retrieval_results).T.round(3)
retrieval_table

,hit@1,hit@5,hit@10,MRR@10
keyword (BM25),0.940,0.982,0.984,0.959
vector baseline,0.967,0.991,0.994,0.977
vector fine-tuned,0.977,0.992,0.994,0.984
hybrid baseline,0.973,0.992,0.995,0.981
hybrid fine-tuned,0.973,0.992,0.996,0.981


## Part 8 — Load the LLM

**Llama-3.2-1B-Instruct**: small enough for a free T4.
- `float16` → half-size numbers, so it uses half the memory.
- `device_map="auto"` → puts it on the GPU automatically.

**If you get an access error:** Llama approval may not be done yet. Swap in `"Qwen/Qwen2.5-1.5B-Instruct"` (no approval needed) and mention it in your write-up.

In [24]:
from transformers import pipeline

LLM_NAME = "meta-llama/Llama-3.2-1B-Instruct"     # fallback: "Qwen/Qwen2.5-1.5B-Instruct"

llm = pipeline("text-generation", model=LLM_NAME,
               torch_dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

### 8.1 The prompt
A **strict prompt** keeps a small model on track: use only the context, reply with one word.
`parse_answer` uses **regex** to pull out yes / no / maybe, in case the model adds extra words.

In [25]:
SYSTEM = ("You are a careful medical research assistant. "
          "If context is provided, base your answer only on that context. "
          "Reply with exactly one word: yes, no, or maybe.")

def build_messages(question, docs=None):
    if docs:
        context = "\n\n".join(f"[{i+1}] {d['text']}" for i, d in enumerate(docs))
        user = f"Context:\n{context}\n\nQuestion: {question}\nAnswer (yes, no, or maybe):"
    else:
        user = f"Question: {question}\nAnswer (yes, no, or maybe):"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def ask_llm(messages):
    out = llm(messages, max_new_tokens=5, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip()

def parse_answer(text):
    m = re.search(r"\b(yes|no|maybe)\b", text.lower())
    return m.group(1) if m else "unknown"

print(ask_llm(build_messages("Is regular exercise associated with lower blood pressure?")))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


no.


## Part 9 — The full RAG function

`mode` lets us switch retrieval on and off:
- `"none"` → LLM answers from memory only (shows why RAG matters)
- `"vector"` → meaning search only
- `"hybrid"` → meaning + keyword

In [26]:
def rag_answer(question, model_name="fine_tuned", mode="hybrid", top_k=3):
    if mode == "none":
        docs = []
    elif mode == "hybrid":
        docs = hybrid_search(question, model_name, k=top_k)
    else:
        docs = vector_search(question, model_name, k=top_k)
    raw = ask_llm(build_messages(question, docs))
    return parse_answer(raw), raw, docs

pred, raw, docs = rag_answer(test_df.question[0])
print("Question:", test_df.question[0])
print("Predicted:", pred, "| Correct:", test_df.label[0])
print("Evidence pubids:", [d["pubid"] for d in docs])

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Predicted: no | Correct: yes
Evidence pubids: ['21645374', '21645374', '18222909']


## Part 10 — End-to-end evaluation

We compare three setups on the same questions:
1. **LLM only** (no retrieval)
2. **RAG + baseline embeddings**
3. **RAG + fine-tuned embeddings**

Start with `N_EVAL = 200` (~3–5 min) to make sure everything works, then set it to **1000** for final numbers.

In [27]:
from tqdm.auto import tqdm

N_EVAL = 200          # change to 1000 for the final run
eval_df = test_df.sample(N_EVAL, random_state=SEED) if N_EVAL < len(test_df) else test_df

setups = {
    "LLM only (no retrieval)":     dict(mode="none"),
    "RAG + baseline embeddings":   dict(model_name="baseline",   mode="hybrid"),
    "RAG + fine-tuned embeddings": dict(model_name="fine_tuned", mode="hybrid"),
}

rows = []
for setup, kwargs in setups.items():
    for _, r in tqdm(eval_df.iterrows(), total=len(eval_df), desc=setup):
        pred, raw, docs = rag_answer(r.question, **kwargs)
        rows.append({"setup": setup, "pubid": r.pubid, "label": r.label, "pred": pred,
                     "gold_retrieved": any(d["pubid"] == r.pubid for d in docs)})

preds = pd.DataFrame(rows)
preds["correct"] = preds.pred == preds.label
preds.head()

LLM only (no retrieval):   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


RAG + baseline embeddings:   0%|          | 0/200 [00:00<?, ?it/s]

RAG + fine-tuned embeddings:   0%|          | 0/200 [00:00<?, ?it/s]

,setup,pubid,label,pred,gold_retrieved,correct
0,LLM only (no retrieval),26348845,yes,no,False,False
1,LLM only (no retrieval),10973547,no,no,False,True
2,LLM only (no retrieval),26163474,yes,no,False,False
3,LLM only (no retrieval),24374414,yes,no,False,False
4,LLM only (no retrieval),18507507,yes,no,False,False


### 10.1 Results tables
- **accuracy** → share of correct answers
- **unknown_rate** → model didn't give yes/no/maybe (a prompt-following problem)
- **gold_in_context** → how often the correct abstract reached the LLM
- **per-class accuracy** → small models usually struggle most with "maybe"

In [28]:
summary = preds.groupby("setup").agg(
    accuracy=("correct", "mean"),
    unknown_rate=("pred", lambda s: (s == "unknown").mean()),
    gold_in_context=("gold_retrieved", "mean"),
).round(3)

per_class = preds.pivot_table(index="setup", columns="label",
                              values="correct", aggfunc="mean").round(3)

display(summary)
display(per_class)

,accuracy,unknown_rate,gold_in_context
setup,,,
LLM only (no retrieval),0.315,0.0,0.00
RAG + baseline embeddings,0.315,0.0,0.98
RAG + fine-tuned embeddings,0.315,0.0,0.98


label,maybe,no,yes
setup,,,
LLM only (no retrieval),0.0,1.0,0.0
RAG + baseline embeddings,0.0,1.0,0.0
RAG + fine-tuned embeddings,0.0,1.0,0.0


### 10.2 The key research link: retrieval quality → answer quality
Compare accuracy when the correct abstract **was** vs **was not** retrieved.
If accuracy is clearly higher when it was retrieved, that supports your research claim: **better retrieval → better, more grounded answers.**

In [29]:
rag_only = preds[preds.setup != "LLM only (no retrieval)"]
rag_only.groupby(["setup", "gold_retrieved"])["correct"].mean().unstack().round(3)

gold_retrieved,False,True
setup,,
RAG + baseline embeddings,0.0,0.321
RAG + fine-tuned embeddings,0.0,0.321


In [30]:
preds.to_csv("rag_predictions.csv", index=False)
retrieval_table.to_csv("retrieval_results.csv")
# Also save to Drive (needs Part 0.5):
preds.to_csv(f"{SAVE_DIR}/rag_predictions.csv", index=False)
retrieval_table.to_csv(f"{SAVE_DIR}/retrieval_results.csv")

## Part 11 — Gradio web app

A small app where anyone can type a question, pick the embedding model and retrieval mode, and see the answer **with the evidence**.
`share=True` gives you a temporary public link to demo.

In [31]:
import gradio as gr

def ui_answer(question, model_name, mode, top_k):
    pred, raw, docs = rag_answer(question, model_name, mode, int(top_k))
    evidence = "\n\n".join(f"**[{i+1}] PubMed {d['pubid']}**\n\n{d['text']}"
                           for i, d in enumerate(docs)) or "_No retrieval used._"
    return pred.upper(), evidence

app = gr.Interface(
    fn=ui_answer,
    inputs=[
        gr.Textbox(label="Medical research question", lines=2),
        gr.Dropdown(["baseline", "fine_tuned"], value="fine_tuned", label="Embedding model"),
        gr.Radio(["hybrid", "vector", "none"], value="hybrid", label="Retrieval mode"),
        gr.Slider(1, 10, value=3, step=1, label="Chunks to retrieve"),
    ],
    outputs=[gr.Textbox(label="Answer"), gr.Markdown(label="Evidence")],
    examples=[[test_df.question[0], "fine_tuned", "hybrid", 3]],
    title="PubMedQA Medical RAG",
    description="Answers yes / no / maybe using retrieved PubMed abstracts.",
)
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d8d9cc8102c4e5653.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
for _, r in test_df.sample(5, random_state=1).iterrows():
    print(r.question)
    print("Correct answer:", r.label)
    print("-" * 60)

Are sports medicine journals relevant and applicable to practitioners and athletes?
Correct answer: no
------------------------------------------------------------
Are patients with serious mental illness more likely to be admitted to nursing homes with more deficiencies in care?
Correct answer: yes
------------------------------------------------------------
Is patient outcome compromised during the initial experience with robot-assisted radical cystectomy?
Correct answer: no
------------------------------------------------------------
Is routine dissection of the station 9 lymph nodes really necessary for primary lung cancer?
Correct answer: no
------------------------------------------------------------
Does solid culture for tuberculosis influence clinical decision making in India?
Correct answer: maybe
------------------------------------------------------------


In [34]:
test_df[test_df.question.str.contains("sports medicine journals")][["pubid", "label"]]

,pubid,label
507,15388567,no


## Part 12 — Write-up checklist

**Fill in your numbers**
- Answer balance (Part 1.2): yes ___ / no ___ / maybe ___
- Retrieval (Part 7.1): hit@1 and MRR@10 for all 5 setups
- End-to-end accuracy (Part 10.1): LLM only ___ vs RAG baseline ___ vs RAG fine-tuned ___
- Accuracy when gold retrieved vs not (Part 10.2)

**README structure**
1. Problem: LLMs hallucinate in medicine; RAG helps only if retrieval is good.
2. Research question: does domain-specific embedding fine-tuning improve retrieval and answers?
3. Data and models (table)
4. Method: chunking → fine-tuning → vector + BM25 + hybrid → LLM
5. Results tables
6. Limitations: small 1B LLM, single dataset, gold abstract is in the corpus, one random seed
7. Next steps

**Experiments to try next (pick 1–2)**
- Epochs 1 vs 3
- Batch size 16 vs 32 vs 64
- `top_k` 1 vs 3 vs 5
- A biomedical model such as PubMedBERT as a third embedding option

---

### Common errors

| Error | Fix |
|---|---|
| `401` / gated repo error | Llama access not approved yet, or HF_TOKEN missing → use the Qwen fallback |
| `CUDA out of memory` | Runtime → Restart session, run again; or lower `batch_size` to 16 |
| `ModuleNotFoundError` | Re-run Part 0.2, then restart the session |
| Colab disconnected | Re-run from the top; fine-tuned model is saved in Drive if you ran Part 4.2 |
| `NameError` | A cell above was skipped; run all cells in order |